In [ ]:
import os
import subprocess
import sys

# Install the course package and all pinned dependencies.
# In GitHub Actions CI this step is skipped (pre-installed via pip install -e .[dev]).
if not os.environ.get("CI"):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/demianw/Xed.git"],
        check=True,
    )

# Data wrangling

**Learning objectives**

By the end of this notebook you will be able to:
1. Create and inspect pandas `DataFrame` and `Series` objects.
2. Select data by label (`.loc`), position (`.iloc`), and boolean masks.
3. Compute descriptive statistics and grouped aggregations (`groupby`).
4. Merge DataFrames from multiple sources.
5. Work with time-indexed data using `resample` and datetime indexing.

This notebook is adapted from Joris Van den Bossche tutorial:

* https://github.com/paris-saclay-cds/python-workshop/blob/master/Day_1_Scientific_Python/02-pandas_introduction.ipynb

In [ ]:
import numpy as np
import pandas as pd

from xed.datasets import (
    load_titanic,
    load_no2,
    load_french_referendum,
    load_french_departments,
    load_french_regions,
    fetch_french_geojson,
)
import matplotlib.pyplot as plt

pd.options.display.max_rows = 8

## 1. Pandas: data analysis in python

For data-intensive work in Python the [Pandas](http://pandas.pydata.org) library has become essential.

**What is `pandas`?**

* Pandas can be thought of as *NumPy arrays with labels* for rows and columns, and better support for heterogeneous data types, but it's also much, much more than that.
* Pandas can also be thought of as `R`'s `data.frame` in Python.
* Powerful for working with missing data, working with time series data, for reading and writing your data, for reshaping, grouping, merging your data, ...

It's documentation: http://pandas.pydata.org/pandas-docs/stable/


**When do you need pandas?**

When working with **tabular or structured data** (like R dataframe, SQL table, Excel spreadsheet, ...):

- Import data
- Clean up messy data
- Explore data, gain insight into data
- Process and prepare your data for analysis
- Analyse your data (together with scikit-learn, statsmodels, ...)

<div class="alert alert-warning">
<b>ATTENTION!</b>: <br><br>

Pandas is great for working with heterogeneous and tabular 1D/2D data, but not all types of data fit in such structures!
<ul>
<li>When working with array data (e.g. images, numerical algorithms): just stick with numpy</li>
<li>When working with multidimensional labeled data (e.g. climate data): have a look at [xarray](http://xarray.pydata.org/en/stable/)</li>
</ul>
</div>

## 2. The pandas data structures: `DataFrame` and `Series`

### 2.1 The 2D table: pandas `DataFrame`

A `DataFrame` is a **tablular data structure** (multi-dimensional object to hold labeled data) comprised of rows and columns, akin to a spreadsheet, database table, or R's data.frame object. You can think of it as multiple Series object which share the same index.


<img align="left" width=50% src="./schema-dataframe.svg">

We can create a pandas Dataframe and specify the index and columns to use.

In [ ]:
data = {
    "country": ["Belgium", "France", "Germany", "Netherlands", "United Kingdom"],
    "population": [11.3, 64.3, 81.3, 16.9, 64.9],
    "area": [30510, 671308, 357050, 41526, 244820],
    "capital": ["Brussels", "Paris", "Berlin", "Amsterdam", "London"],
}
df_countries = pd.DataFrame(data)
df_countries

We can check that we are manipulating a Pandas DataFrame

In [ ]:
type(df_countries)

As previously mentioned, the dataframe stores information regarding the column and index information.

In [ ]:
df_countries.columns

In [ ]:
df_countries.index

You can get an overview of the information of a dataframe using the `info()` method:

In [ ]:
df_countries.info()

An information which is quite useful is related to the data type.

It is important to know that machine learning algorithms are based on mathematics and algebra. Thus, these algorithms expect numerical data.

Pandas allows to read, manipulate, explore, and transform heterogeneous data to numerical data.

In [ ]:
df_countries.dtypes

#### Exercise

We will define a set of 1D NumPy arrays containing the data that we will work with.

In [ ]:
country_name = ["Austria", "Iran, Islamic Rep.", "France"]
country_code = ["AUT", "IRN", "FRA"]
gdp_2015 = [1349034029453.37, 385874474398.59, 2438207896251.84]
gdp_2017 = [1532397555.55556, 439513511620.591, 2582501307216.42]

* Create a Python dictionary where the keys will be the name of the columns and the values will be the corresponding Python list.

In [ ]:
# %cat solutions/02_solutions.py

* Use the same procedure (Python dictionary) but specify that the country code should be used as the index. Therefore, check the parameter `index_col` or the method `DataFrame.set_index()`

In [ ]:
# %cat solutions/03_solutions.py

In [ ]:
# %cat solutions/04_solutions.py

### 2.2 One-dimensional data: `Series` (a column of a DataFrame)

A Series is a basic holder for **one-dimensional labeled data**.

In [ ]:
df_countries

In [ ]:
df_countries.loc[:, "population"]

In [ ]:
population = df_countries.loc[:, "population"]

We can check that we manipulate a Pandas Series

In [ ]:
type(population)

### 2.3 Data import and export

A wide range of input/output formats are natively supported by pandas:

* CSV, text
* SQL database
* Excel
* HDF5
* json
* html
* pickle
* sas, stata
* (parquet)
* ...

In [ ]:
# pd.read_xxxx

In [ ]:
# df.to_xxxx

Very powerful csv reader:

In [ ]:
pd.read_csv?

Luckily, if we have a well formed csv file, we don't need many of those arguments:

In [ ]:
import os

In [ ]:
df = load_titanic()

In [ ]:
df.head()

In [ ]:
df.info()

<div class="alert alert-success">

<b>EXERCISE</b>: Load the Belgian NO₂ air quality dataset into a DataFrame `no2`
using <code>load_no2()</code> from <code>xed.datasets</code>.
<br><br>
Some aspects about the file:
 <ul>
  <li>Which separator is used in the file?</li>
  <li>The second row includes unit information and should be skipped (check `skiprows` keyword)</li>
  <li>For missing values, it uses the `'n/d'` notation (check `na_values` keyword)</li>
  <li>We want to parse the 'timestamp' column as datetimes (check the `parse_dates` keyword)</li>
</ul>
</div>

In [ ]:
# %cat solutions/22_solutions.py

In [ ]:
no2.info()

## 3. Selecting and filtering data

One of pandas' basic features is the labeling of rows and columns, but this makes indexing a bit complex. We now have to distinguish between:

* selection by **label**
* selection by **position**


### 3.1 Indexing by label using `.loc`

We will first select data from the dataframe selecting by **label**.

In [ ]:
data = {
    "country": ["Belgium", "France", "Germany", "Netherlands", "United Kingdom"],
    "population": [11.3, 64.3, 81.3, 16.9, 64.9],
    "area": [30510, 671308, 357050, 41526, 244820],
    "capital": ["Brussels", "Paris", "Berlin", "Amsterdam", "London"],
}
df_countries = pd.DataFrame(data).set_index("country")
df_countries

The syntax to select by label is `.loc['row_name', 'col_name']`. Therefore, we can get a row of the dataframe by indicating the name of the index to select.

In [ ]:
df_countries.loc["France", :]

Similarly, we can get a column of the dataframe by indicating the name of the column.

In [ ]:
df_countries.loc[:, "area"]

Specifying both index and column name, we will get the intersection of the row and the column.

In [ ]:
df_countries.loc["France", "area"]

We can get several columns by passing a list of the columns to be selected.

In [ ]:
x = df_countries.loc["France", ["area", "population"]]

This is the exact same behavior with the index for the rows.

In [ ]:
df_countries.loc[["France", "Belgium"], ["area", "population"]]

You can go further and slice a portion of the dataframe.

In [ ]:
df_countries.loc["France":"Netherlands", :]

Note that in this case, the first and last item of the slice are selected.

### 3.2  Indexing by position using `.iloc`

Sometimes, it is handy to select a portion of the data given the row and column indices number. We can this indexing by **position**.

In [ ]:
data = {
    "country": ["Belgium", "France", "Germany", "Netherlands", "United Kingdom"],
    "population": [11.3, 64.3, 81.3, 16.9, 64.9],
    "area": [30510, 671308, 357050, 41526, 244820],
    "capital": ["Brussels", "Paris", "Berlin", "Amsterdam", "London"],
}
df_countries = pd.DataFrame(data).set_index("country")
df_countries

The syntax is similar to `.loc`. It will be `.iloc[row_id, col_id]`. We can get the first row.

In [ ]:
df_countries.iloc[0, :]

Or the last column.

In [ ]:
df_countries.iloc[:, -1]

And make the intersections.

In [ ]:
df_countries.iloc[0, -1]

Passing a list of indices is also working.

In [ ]:
df_countries.iloc[[0, 1], [-2, -1]]

And we can use slicing as well.

In [ ]:
df_countries.iloc[1:3, 0:2]

However, be aware that the ending index of the slice is discarded.

### 3.3 Use the pandas shortcut

In [ ]:
data = {
    "country": ["Belgium", "France", "Germany", "Netherlands", "United Kingdom"],
    "population": [11.3, 64.3, 81.3, 16.9, 64.9],
    "area": [30510, 671308, 357050, 41526, 244820],
    "capital": ["Brussels", "Paris", "Berlin", "Amsterdam", "London"],
}
df_countries = pd.DataFrame(data).set_index("country")
df_countries

Pandas provides a shortcut to select some part of the data.

In [ ]:
df_countries["population"]

In [ ]:
df_countries[["area", "capital"]]

In [ ]:
df_countries[2:5]

In [ ]:
df_countries["Germany":"United Kingdom"]

You don't need to use `loc` and `iloc`. The selection rules are:

* Passing a single label or list of labels will select a column or several columns;
* Passing a slice (label or indices) will select the corresponding rows.

You can always use the systematic indexing to avoid confusion. Use the shortcut at your own risk.

### 3.4 Boolean indexing (filtering)

Often, you want to select rows based on a certain condition. This can be done with 'boolean indexing' (like a where clause in SQL) and comparable to numpy.

The indexer (or boolean mask) should be 1-dimensional and the same length as the thing being indexed.

In [ ]:
df_countries["population"] > 60

In [ ]:
mask_pop_above_60 = df_countries["population"] > 60

We can then use this mask to index a series or a DataFrame.

In [ ]:
population = df_countries["population"]

In [ ]:
population.loc[mask_pop_above_60]

In [ ]:
population[mask_pop_above_60]

In [ ]:
df_countries.loc[mask_pop_above_60]

In [ ]:
df_countries[~mask_pop_above_60]

### 3.5 Exercise

In [ ]:
df = load_titanic()
df.head()

Select the sub-dataframe for which the men are older than 60 years old. Using the attribute shape, find how many individual correspond to this criteria.

In [ ]:
# %cat solutions/05_solutions.py

## 4. Statistical analysis

Pandas provides an easy and fast way to explore data. Let's explore the `titanic` data set.

In [ ]:
df = df.set_index("Name")
df.head()

We will select the `Age` column and compute couple of statistic.

In [ ]:
age = df["Age"]
age

In [ ]:
age.mean()

In [ ]:
age.max()

In [ ]:
age.min()

In [ ]:
age.describe()

In [ ]:
age.value_counts()

In [ ]:
age.hist(bins=100)

### Exercise

* What is the maximum Fare that was paid? And the median?

In [ ]:
# %cat solutions/06_solutions.py

In [ ]:
# %cat solutions/07_solutions.py

* Calculate the average survival ratio for passengers (note: the 'Survived' column indicates whether someone survived (1) or not (0)).

In [ ]:
# %cat solutions/08_solutions.py

In [ ]:
# %cat solutions/09_solutions.py

* Select the sub-dataframe for which the men are older than 60 years old.

In [ ]:
# %cat solutions/10_solutions.py

* Based on the titanic data set, select all rows for male passengers and calculate the mean age of those passengers. Do the same for the female passengers.

In [ ]:
# %cat solutions/11_solutions.py

In [ ]:
# %cat solutions/12_solutions.py

* Plot the Fare distribution.

In [ ]:
# %cat solutions/13_solutions.py

## 5. The group-by operation

### Some 'theory': the groupby operation (split-apply-combine)

In [ ]:
df = pd.DataFrame(
    {
        "key": ["A", "B", "C", "A", "B", "C", "A", "B", "C"],
        "data": [0, 5, 10, 5, 10, 15, 10, 15, 20],
    }
)
df

### 5.1 Recap: aggregating functions

When analyzing data, you often calculate summary statistics (aggregations like the mean, max, ...). As we have seen before, we can easily calculate such a statistic for a Series or column using one of the many available methods. For example:

In [ ]:
df["data"].sum()

However, in many cases your data has certain groups in it, and in that case, you may want to calculate this statistic for each of the groups.

For example, in the above dataframe `df`, there is a column 'key' which has three possible values: 'A', 'B' and 'C'. When we want to calculate the sum for each of those groups, we could do the following:

In [ ]:
for key in ["A", "B", "C"]:
    print(key, df[df["key"] == key]["data"].sum())

This becomes very verbose when having multiple groups. You could make the above a bit easier by looping over the different values, but still, it is not very convenient to work with.

What we did above, applying a function on different groups, is a "groupby operation", and pandas provides some convenient functionality for this.

### 5.2 Groupby: applying functions per group

The "group by" concept: we want to **apply the same function on subsets of your dataframe, based on some key to split the dataframe in subsets**

This operation is also referred to as the "split-apply-combine" operation, involving the following steps:

* **Splitting** the data into groups based on some criteria
* **Applying** a function to each group independently
* **Combining** the results into a data structure

<img src="./splitApplyCombine.png">

Similar to SQL `GROUP BY`

Instead of doing the manual filtering as above


    df[df['key'] == "A"].sum()
    df[df['key'] == "B"].sum()
    ...

pandas provides the `groupby` method to do exactly this:

In [ ]:
df.groupby("key").sum()

In [ ]:
df.groupby("key").aggregate([np.sum, np.median])  # 'sum'

And many more methods are available.

In [ ]:
df.groupby("key")["data"].sum()

In [ ]:
for group_name, group_df in df.groupby("key"):
    print(group_name)
    print(group_df)

### 5.3 Exercise: Application of the groupby concept on the titanic data

We go back to the titanic passengers survival data:

In [ ]:
df = load_titanic()
df = df.set_index("Name")

In [ ]:
df.head()

* Using `groupby()`, calculate the average age for each sex.</li>


In [ ]:
# %cat solutions/14_solutions.py

* Using the `groupby()` function, plot the age distribution for each sex.

In [ ]:
# %cat solutions/15_solutions.py

* Plot the fare distribution based on the class.

In [ ]:
# %cat solutions/16_solutions.py

* Plot the survival rate by class with a bar plot.

In [ ]:
# %cat solutions/17_solutions.py

* Compute the survival rate grouping by class and sex. (Hint: you can pass a list to the `groupby` function)

In [ ]:
# %cat solutions/18_solutions.py

## 7. Merging different source of information

### 7.1 Simple concatenation

In [ ]:
# series
population = pd.Series(
    {"Germany": 81.3, "Belgium": 11.3, "France": 64.3, "United Kingdom": 64.9, "Netherlands": 16.9}
)

# dataframe
data = {
    "country": ["Belgium", "France", "Germany", "Netherlands", "United Kingdom"],
    "population": [11.3, 64.3, 81.3, 16.9, 64.9],
    "area": [30510, 671308, 357050, 41526, 244820],
    "capital": ["Brussels", "Paris", "Berlin", "Amsterdam", "London"],
}
countries = pd.DataFrame(data)
countries

Assume we have some similar data as in countries, but for a set of different countries:

In [ ]:
data = {
    "country": [
        "Nigeria",
        "Rwanda",
        "Egypt",
        "Morocco",
    ],
    "population": [182.2, 11.3, 94.3, 34.4],
    "area": [923768, 26338, 1010408, 710850],
    "capital": ["Abuja", "Kigali", "Cairo", "Rabat"],
}
countries_africa = pd.DataFrame(data)
countries_africa

We now want to combine the rows of both datasets:

In [ ]:
pd.concat([countries, countries_africa])

If we don't want the index to be preserved:

In [ ]:
pd.concat([countries, countries_africa], ignore_index=True)

When the two dataframes don't have the same set of columns, by default missing values get introduced:

In [ ]:
pd.concat([countries_africa[["country", "capital"]], countries], ignore_index=True, sort=False)

## 7.2 Combining columns instead of rows

Assume we have another DataFrame for the same countries, but with some additional statistics:

In [ ]:
data = {
    "country": ["Belgium", "France", "Netherlands"],
    "GDP": [496477, 2650823, 820726],
    "area": [8.0, 9.9, 5.7],
}
country_economics = pd.DataFrame(data).set_index("country")
country_economics

In [ ]:
pd.concat([countries, country_economics], axis=1)

`pd.concat` matches the different objects based on the index:

In [ ]:
countries2 = countries.set_index("country")

In [ ]:
countries2

In [ ]:
pd.concat([countries2, country_economics], axis=1, sort=False)

### 7.3 Dataframe merging

Using `pd.concat` above, we combined datasets that had the same columns or the same index values. But, another typical case if where you want to add information of second dataframe to a first one based on one of the columns. That can be done with `pd.merge`.

Let's look again at the titanic passenger data, but taking a small subset of it to make the example easier to grasp:

In [ ]:
df = load_titanic()
df = df.loc[:9, ["Survived", "Pclass", "Sex", "Age", "Fare", "Embarked"]]

In [ ]:
df

Assume we have another dataframe with more information about the 'Embarked' locations:

In [ ]:
locations = pd.DataFrame(
    {
        "Embarked": ["S", "C", "Q", "N"],
        "City": ["Southampton", "Cherbourg", "Queenstown", "New York City"],
        "Country": ["United Kindom", "France", "Ireland", "United States"],
    }
)

In [ ]:
locations

We now want to add those columns to the titanic dataframe, for which we can use `pd.merge`, specifying the column on which we want to merge the two datasets:

In [ ]:
pd.merge(df, locations, on="Embarked", how="left")

In this case we use `how='left'` (a "left join") because we wanted to keep the original rows of df and only add matching values from locations to it. Other options are 'inner', 'outer' and 'right' (see the docs for more on this).

## 8. Working with time series data

### 8.1 Time series preamble

In [ ]:
no2 = load_no2()

When we ensure the DataFrame has a `DatetimeIndex`, time-series related functionality becomes available:

In [ ]:
no2.index

Indexing a time series works with strings:

In [ ]:
no2["2010-01-01 09:00":"2010-01-01 12:00"]

A nice feature is "partial string" indexing, so you don't need to provide the full datetime string.

E.g. all data of January up to March 2012:

In [ ]:
no2["2012-01":"2012-03"]

Time and date components can be accessed from the index:

In [ ]:
no2.index.hour

In [ ]:
no2.index.year

### 8.2 The power of pandas: `resample`

A very powerful method is **`resample`: converting the frequency of the time series** (e.g. from hourly to daily data).

Remember the air quality data:

In [ ]:
no2.plot()

The time series has a frequency of 1 hour. I want to change this to daily:

In [ ]:
no2.head()

In [ ]:
no2.resample("D").mean().head()

Above I take the mean, but as with `groupby` I can also specify other methods:

In [ ]:
no2.resample("D").max().head()

The string to specify the new time frequency: http://pandas.pydata.org/pandas-docs/dev/timeseries.html#offset-aliases
These strings can also be combined with numbers, eg `'10D'`.

Further exploring the data:

In [ ]:
no2.resample("M").mean().plot()  # 'A'

In [ ]:
no2.loc["2009":, "VERS"].resample("M").agg(["mean", "median"]).plot()

### 8.3 Exercise

The evolution of the yearly averages with, and the overall mean of all stations

* Use `resample` and `plot` to plot the yearly averages for the different stations.
* The overall mean of all stations can be calculated by taking the mean of the different columns (`.mean(axis=1)`).



## Further reading

* Pandas documentation: http://pandas.pydata.org/pandas-docs/stable/

* Books

    * "Python for Data Analysis" by Wes McKinney
    * "Python Data Science Handbook" by Jake VanderPlas

* Tutorials (many good online tutorials!)

  * https://github.com/jorisvandenbossche/pandas-tutorial
  * https://github.com/brandon-rhodes/pycon-pandas-tutorial

* Tom Augspurger's blog

  * https://tomaugspurger.github.io/modern-1.html